# 09: Analytics Connectors

This notebook demonstrates the analytics connector integrations in siege_utilities:

1. **Facebook Business** - Ads, Pages, Business Manager
2. **Google Analytics** - GA4 reporting
3. **Snowflake** - Data warehouse queries
4. **data.world** - Dataset discovery and access

## Prerequisites

```bash
pip install facebook-business google-analytics-data datadotworld snowflake-connector-python
```

## Environment Variables

Set these before running:
```bash
# Facebook Business
export FB_ACCESS_TOKEN="your-token"
export FB_APP_ID="your-app-id"  # optional
export FB_APP_SECRET="your-secret"  # optional

# Google Analytics
export GOOGLE_APPLICATION_CREDENTIALS="/path/to/service-account.json"

# Snowflake
export SNOWFLAKE_ACCOUNT="your-account"
export SNOWFLAKE_USER="your-user"
export SNOWFLAKE_PASSWORD="your-password"
export SNOWFLAKE_WAREHOUSE="your-warehouse"

# data.world
export DW_AUTH_TOKEN="your-token"
```

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

# Check which connectors are available
print("=== Connector Availability ===")

connectors = {
    'Facebook Business': ('FB_ACCESS_TOKEN', 'facebook_business'),
    'Google Analytics': ('GOOGLE_APPLICATION_CREDENTIALS', 'google.analytics.data_v1beta'),
    'Snowflake': ('SNOWFLAKE_ACCOUNT', 'snowflake.connector'),
    'data.world': ('DW_AUTH_TOKEN', 'datadotworld')
}

for name, (env_var, module) in connectors.items():
    has_creds = bool(os.environ.get(env_var))
    try:
        __import__(module)
        has_lib = True
    except ImportError:
        has_lib = False
    
    status = "Ready" if (has_creds and has_lib) else f"Missing: {'' if has_lib else 'library '}{'' if has_creds else 'credentials'}"
    print(f"  {name}: {status}")

## 1. Facebook Business Connector

Connect to Facebook Ads API to retrieve ad performance data.

In [ ]:
# Facebook Business Connector
from siege_utilities.analytics.facebook_business import FacebookBusinessConnector

# Get credentials from environment
fb_token = os.environ.get('FB_ACCESS_TOKEN')
fb_app_id = os.environ.get('FB_APP_ID')
fb_app_secret = os.environ.get('FB_APP_SECRET')

if fb_token:
    print("Initializing Facebook Business connector...")
    fb = FacebookBusinessConnector(
        access_token=fb_token,
        app_id=fb_app_id,
        app_secret=fb_app_secret
    )
    print("Connected!")
else:
    print("FB_ACCESS_TOKEN not set. Set it and re-run this cell.")
    fb = None

In [ ]:
# List accessible ad accounts
if fb:
    print("Retrieving ad accounts...")
    accounts = fb.get_ad_accounts()
    
    if accounts:
        print(f"\nFound {len(accounts)} ad account(s):")
        for acc in accounts:
            print(f"  - {acc['name']} (ID: {acc['id']})")
            print(f"    Currency: {acc.get('currency', 'N/A')}, Status: {acc.get('account_status', 'N/A')}")
    else:
        print("No ad accounts found or access denied.")

In [ ]:
# Get ad insights (requires an ad account ID)
if fb and accounts:
    from datetime import datetime, timedelta
    
    # Use first account
    account_id = accounts[0]['id']
    
    # Last 30 days
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')
    
    print(f"Retrieving insights for {accounts[0]['name']}...")
    print(f"Date range: {start_date} to {end_date}")
    
    insights = fb.get_ad_insights(
        ad_account_id=account_id,
        start_date=start_date,
        end_date=end_date,
        fields=['impressions', 'clicks', 'spend', 'cpc', 'ctr']
    )
    
    if not insights.empty:
        print(f"\nRetrieved {len(insights)} rows of insights:")
        print(insights.head())
    else:
        print("No insights data available for this period.")

## 2. Google Analytics Connector

Connect to GA4 to retrieve website analytics data.

In [ ]:
# Google Analytics Connector
from siege_utilities.analytics.google_analytics import GoogleAnalyticsConnector

ga_creds_path = os.environ.get('GOOGLE_APPLICATION_CREDENTIALS')

if ga_creds_path and os.path.exists(ga_creds_path):
    import json
    with open(ga_creds_path) as f:
        service_account_data = json.load(f)
    
    print("Initializing Google Analytics connector...")
    ga = GoogleAnalyticsConnector(
        auth_method='service_account',
        service_account_data=service_account_data
    )
    print("Connected!")
else:
    print("GOOGLE_APPLICATION_CREDENTIALS not set or file not found.")
    print("Set it to path of your service account JSON file.")
    ga = None

In [ ]:
# Get GA4 report (requires property ID)
if ga:
    # Replace with your GA4 property ID
    PROPERTY_ID = "properties/123456789"  # <-- UPDATE THIS
    
    from datetime import datetime, timedelta
    
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')
    
    print(f"Retrieving GA4 data for {PROPERTY_ID}...")
    
    report = ga.get_ga4_report(
        property_id=PROPERTY_ID,
        start_date=start_date,
        end_date=end_date,
        metrics=['sessions', 'activeUsers', 'screenPageViews'],
        dimensions=['date', 'country']
    )
    
    if report is not None and not report.empty:
        print(f"\nRetrieved {len(report)} rows:")
        print(report.head(10))
    else:
        print("No data returned.")

## 3. Snowflake Connector

Connect to Snowflake data warehouse for SQL queries.

In [ ]:
# Snowflake Connector
from siege_utilities.analytics.snowflake_connector import get_snowflake_connector

sf_account = os.environ.get('SNOWFLAKE_ACCOUNT')
sf_user = os.environ.get('SNOWFLAKE_USER')
sf_password = os.environ.get('SNOWFLAKE_PASSWORD')
sf_warehouse = os.environ.get('SNOWFLAKE_WAREHOUSE')

if all([sf_account, sf_user, sf_password]):
    print("Initializing Snowflake connector...")
    sf = get_snowflake_connector(
        account=sf_account,
        user=sf_user,
        password=sf_password,
        warehouse=sf_warehouse
    )
    print("Connected!")
else:
    print("Snowflake credentials not set. Required:")
    print("  SNOWFLAKE_ACCOUNT, SNOWFLAKE_USER, SNOWFLAKE_PASSWORD")
    sf = None

In [ ]:
# Run a Snowflake query
if sf:
    query = "SELECT CURRENT_TIMESTAMP(), CURRENT_USER(), CURRENT_WAREHOUSE()"
    
    print("Running test query...")
    result = sf.query(query)
    
    if result is not None:
        print("Query successful!")
        print(result)

## 4. data.world Connector

Search and access datasets from data.world.

In [ ]:
# data.world Connector
from siege_utilities.analytics.datadotworld_connector import DataDotWorldConnector

dw_token = os.environ.get('DW_AUTH_TOKEN')

if dw_token:
    print("Initializing data.world connector...")
    dw = DataDotWorldConnector(api_token=dw_token)
    print("Connected!")
else:
    print("DW_AUTH_TOKEN not set.")
    print("Get your token from: https://data.world/settings/advanced")
    dw = None

In [ ]:
# Search for datasets
if dw:
    print("Searching for 'census' datasets...")
    results = dw.search_datasets(query='census', limit=5)
    
    if results:
        print(f"\nFound {len(results)} datasets:")
        for ds in results:
            print(f"  - {ds.get('title', 'Untitled')}")
            print(f"    Owner: {ds.get('owner', 'Unknown')}")
    else:
        print("No datasets found.")

In [ ]:
# Load a specific dataset
if dw:
    # Example: load a public dataset
    dataset_key = "census/us-population-by-zip-code"  # <-- UPDATE THIS
    
    try:
        print(f"Loading dataset: {dataset_key}...")
        df = dw.load_dataset(dataset_key)
        
        if df is not None:
            print(f"\nLoaded {len(df)} rows, {len(df.columns)} columns")
            print(df.head())
    except Exception as e:
        print(f"Could not load dataset: {e}")

## Summary

| Connector | Class | Key Methods |
|-----------|-------|-------------|
| Facebook Business | `FacebookBusinessConnector` | `get_ad_accounts()`, `get_ad_insights()` |
| Google Analytics | `GoogleAnalyticsConnector` | `get_ga4_report()`, `get_accounts()` |
| Snowflake | `get_snowflake_connector()` | `query()`, `upload_to_snowflake()` |
| data.world | `DataDotWorldConnector` | `search_datasets()`, `load_dataset()` |

### Credential Setup Summary

| Connector | Environment Variable | How to Get |
|-----------|---------------------|------------|
| Facebook | `FB_ACCESS_TOKEN` | [developers.facebook.com](https://developers.facebook.com) → Graph API Explorer |
| Google Analytics | `GOOGLE_APPLICATION_CREDENTIALS` | [console.cloud.google.com](https://console.cloud.google.com) → Service Account |
| Snowflake | `SNOWFLAKE_ACCOUNT`, `_USER`, `_PASSWORD` | Snowflake admin console |
| data.world | `DW_AUTH_TOKEN` | [data.world/settings/advanced](https://data.world/settings/advanced) |